In [1]:
import os
import pandas as pd
import json
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math 
import random 
import igraph as ig
import networkx as nx
from math import log
import pickle
import gzip
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
import copy
import community
import sys 
from pathlib import Path 

import

In [2]:

ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parents[0]
SRC  = ROOT / "src"
DATA = ROOT / "data"
FIGS = ROOT / "figures"

print("Project root:", ROOT)
print("Data folder:", DATA)
print("Figures folder:", FIGS)


#import functions and libraries:
# Permette di importare moduli da src/
sys.path.insert(0, str(SRC))
from utils import * 
from distance_in_network import *

Project root: /Users/Daniele/Downloads/PolitoSphere Finale (Postzip)
Data folder: /Users/Daniele/Downloads/PolitoSphere Finale (Postzip)/data
Figures folder: /Users/Daniele/Downloads/PolitoSphere Finale (Postzip)/figures


Graph

In [3]:
edgelist_df = pd.read_csv(DATA / "interaction_edgelist_ex.csv") #validated subreddit network (2013)
print(edgelist_df.head())
G = nx.from_pandas_edgelist(edgelist_df, source="source", target="target")
print("Graph loaded with", G.number_of_nodes(), "nodes and", G.number_of_edges(), "edges.")


                source                 target
0  PoliticalDiscussion            progressive
1  PoliticalDiscussion              democrats
2  PoliticalDiscussion       askaconservative
3  PoliticalDiscussion  politicalfactchecking
4  PoliticalDiscussion             uspolitics
Graph loaded with 167 nodes and 1395 edges.


Node label

In [4]:
csvdf2 = pd.read_csv(DATA / 'Subreddit_Tags.csv', sep=';', index_col=False)

Color_Field_df = pd.read_csv(DATA / 'Tag_Color.csv')

# ricrea i dizionari
Color_Field = dict(zip(Color_Field_df["Tag"], Color_Field_df["Color"]))

DizValTag = {} 

for i in range(len(csvdf2)) : 
    nome=csvdf2['subreddit'][i]
    n = 0
    for col in csvdf2.columns : 
        if col != 'subreddit' : 
            if csvdf2[col][i] != 0 : 
                n += 1 
    for col in csvdf2.columns : 
        if col != 'subreddit' : 
            if csvdf2[col][i] == 1 : 
                DizValTag[col+"_"+nome] = (1.0/n)
print(len(DizValTag)) 
print(DizValTag)

dem_nodes, cons_nodes, ban_nodes = [], [], []

for sub in G.nodes():
    if f"Dem_{sub}" in DizValTag and DizValTag[f"Dem_{sub}"] != 0:
        dem_nodes.append(sub)
    if f"Cons_{sub}" in DizValTag and DizValTag[f"Cons_{sub}"] != 0:
        cons_nodes.append(sub)
    if f"Ban_{sub}" in DizValTag and DizValTag[f"Ban_{sub}"] != 0:
        ban_nodes.append(sub)

print(f"Dem nodes: {len(dem_nodes)} | Cons nodes: {len(cons_nodes)} | Ban nodes: {len(ban_nodes)}")



715
{'Dem_2012Elections': 0.5, 'Cons_2012Elections': 0.5, 'Dem_2016Elections': 0.5, 'Cons_2016Elections': 0.5, 'Dem_2016_elections': 0.5, 'Cons_2016_elections': 0.5, 'Lib_2ALiberals': 0.5, 'Gun_2ALiberals': 0.5, 'Politic_AOC': 0.5, 'Dem_AOC': 0.5, 'SocialJustice_Abortiondebate': 1.0, 'Geop_ActiveMeasures': 0.5, 'News_ActiveMeasures': 0.5, 'SocialJustice_AgainstHateSubreddits': 1.0, 'Ban_AgainstTheChimpire': 1.0, 'Econ_Agorism': 1.0, 'Canada_Albertapolitics': 1.0, 'Dem_AlexandriaOcasio': 1.0, 'Ban_AltRightChristian': 1.0, 'Politic_AmalaNetwork': 1.0, 'Politic_AmericanPolitics': 1.0, 'Far-Right_AnCap101': 1.0, 'Far-Left_Anarchism': 1.0, 'Far-Left_AnarchismOnline': 1.0, 'News_AnarchistNews': 0.5, 'Far-Left_AnarchistNews': 0.5, 'Far-Left_Anarcho_Capitalism': 1.0, 'Far-Left_Anarchy101': 1.0, 'Far-Right_AntiSemitismInReddit': 1.0, 'Dem_AntiTrumpAlliance': 1.0, 'Far-Right_AnticommieCringe': 1.0, 'Far-Left_AntifascistsofReddit': 1.0, 'Dem_AnybodyButHillary': 1.0, 'News_AnythingGoesNews': 1.0, 

Distances

In [5]:
total_mean, total_err = harmonic_mean_distance(G, list(G.nodes()))
print(f"Total harmonic mean distance: {total_mean:.4f} ± {total_err:.4f}")

# Dem–Cons
dem_cons_mean, dem_cons_err = harmonic_mean_distance(G, dem_nodes, cons_nodes)
dem_cons_norm = dem_cons_mean / total_mean if total_mean else 0
print(f"Dem–Cons distance (norm): {dem_cons_norm:.4f} ± {dem_cons_err:.4f}")

# Dem–Ban
dem_ban_mean, dem_ban_err = harmonic_mean_distance(G, dem_nodes, ban_nodes)
dem_ban_norm = dem_ban_mean / total_mean if total_mean else 0
print(f"Dem–Ban distance (norm): {dem_ban_norm:.4f} ± {dem_ban_err:.4f}")

# Cons–Ban
cons_ban_mean, cons_ban_err = harmonic_mean_distance(G, cons_nodes, ban_nodes)
cons_ban_norm = cons_ban_mean / total_mean if total_mean else 0
print(f"Cons–Ban distance (norm): {cons_ban_norm:.4f} ± {cons_ban_err:.4f}")


Total harmonic mean distance: 2.2685 ± 0.0019
Dem–Cons distance (norm): 0.9217 ± 0.0249
Dem–Ban distance (norm): 1.3043 ± 0.0231
Cons–Ban distance (norm): 0.9618 ± 0.0205
